# 02 · 物理资产构建

模块一原始重建 → VLM 物性先验 → 视频观测检查 → 自动碰撞代理优化与仿真检查 → USD 导出。

视觉表面、部件位姿和 start/end 范围来自模块一。重建缺口如实展示；本模块只自动优化碰撞代理。


In [ ]:
import json
import uuid

from examples._display import setup_notebook, show_animation, show_image, show_metrics

PROJECT_ROOT, OUTPUT_DIR = setup_notebook("physical_asset_building")

from physrealx import ArtifactRef, AssetSpec, PhysRealX, VersionedRef
from physrealx.integrations.physical_estimation import inspect_video
from physrealx.integrations.reconstructed_asset import reconstructed_motion

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
run_key = uuid.uuid4().hex[:12]


## 1. 资产与视频

In [ ]:
upstream = json.loads(
    (PROJECT_ROOT / "examples/real_assets_and_digital_twin/output/results.json").read_text()
)
reference_model = ArtifactRef.model_validate(upstream["reference-model.json"])
reconstruction_report = ArtifactRef.model_validate(upstream["reconstruction-report.json"])
registration = ArtifactRef.model_validate(upstream["reference-registration.json"])
reference = json.loads(px.artifacts.get_bytes(reference_model))
registration_document = json.loads(px.artifacts.get_bytes(registration))
reconstruction_document = json.loads(px.artifacts.get_bytes(reconstruction_report))
expected_reconstructed_motion = reconstructed_motion(
    reconstruction_document, registration_document, reference
)

saved_prior_path = OUTPUT_DIR / "vlm-physics-prior.json"
reconstructed_geometry = None
if saved_prior_path.exists():
    try:
        cached_prior = json.loads(saved_prior_path.read_text())
        candidate = ArtifactRef.model_validate(cached_prior["source_geometry"])
        cached_geometry = (
            json.loads(px.artifacts.get_bytes(candidate)) if px.artifacts.verify(candidate) else {}
        )
        if (
            cached_geometry.get("schema") == "physrealx.registered-reconstruction-geometry/4"
            and cached_geometry.get("geometry_policy") == "coordinate_transform_only"
            and cached_geometry.get("source_reconstruction")
            == reconstruction_report.model_dump(mode="json")
            and cached_geometry.get("source_registration") == registration.model_dump(mode="json")
            and cached_geometry.get("source_reference") == reference_model.model_dump(mode="json")
            and cached_geometry.get("pose_policy") == "module_one_predicted_start_pose_locked"
            and cached_geometry.get("reconstructed_motion") == expected_reconstructed_motion
            and set(cached_geometry.get("source_pose_reprojection_error_m", {}))
            == {link["name"] for link in reference["links"]}
            and max(cached_geometry["source_pose_reprojection_error_m"].values())
            <= 1e-9
        ):
            reconstructed_geometry = candidate
    except (OSError, ValueError, KeyError):
        pass
if reconstructed_geometry is None:
    reconstructed_geometry = px.assets.register_reconstruction_geometry(
        reconstruction_report,
        registration,
    )
real_geometry = json.loads(px.artifacts.get_bytes(reconstructed_geometry))
capture_dir = px.assets.resolve_observation_source(
    VersionedRef.model_validate(reference["source_observations"])
)
video_dir = capture_dir / "real/video"
readiness = inspect_video(px.artifacts, video_dir)
video_status = json.loads(px.artifacts.get_bytes(readiness))
show_metrics(
    {
        **video_status["counts"],
        "几何来源": real_geometry["geometry_source"],
        "部件位姿": "锁定功能一重建 start 状态",
        "最大位姿回投误差 (nm)": 1e9
        * max(real_geometry["source_pose_reprojection_error_m"].values()),
        "视觉表面": "模块一原始重建；仅转换坐标系",
        "功能一预测 start/end (rad)": expected_reconstructed_motion["positions"],
        "碰撞验收锁定范围 (rad)": expected_reconstructed_motion["range"],
        "三角面与颜色": "原样保留",
        "部件顶点 / 三角面": ", ".join(
            f"{link}: {values['vertices']} / {values['triangles']}"
            for link, values in real_geometry["links"].items()
        ),
    }
)


## 2. VLM 物性先验

方法：VLM（Qwen3-VL）。


In [ ]:
images = tuple(
    px.artifacts.put_file(video_dir / "img" / f"{index}.png", media_type="image/png")
    for index in (0, 46, 93, 139, 185)
)
prior = px.assets.predict_physics(
    reference_model,
    registration,
    reconstructed_geometry,
    images,
    python="/home/yxp/miniconda3/envs/qwen/bin/python",
    model="/mnt/53dad72e-a787-4c09-961e-eff73e08888e/yxp/Desktop/ArtCot/models/Qwen3-VL-8B-Instruct",
    revision="local-content-hashed",
    timeout_seconds=1200,
    cached_prior=px.artifacts.put_file(saved_prior_path, media_type="application/json")
    if saved_prior_path.is_file()
    else None,
)
# Save the validated prior at this stage so a collision retry can reuse it.
prior_payload = px.artifacts.get_bytes(prior)
prior_temporary = saved_prior_path.with_suffix(".tmp")
prior_temporary.write_bytes(prior_payload)
prior_temporary.replace(saved_prior_path)
prediction = json.loads(prior_payload)
show_metrics(
    {
        name + " mass (kg)": (
            f"{body['mass_kg']['value']} [{body['mass_kg']['lower']}, {body['mass_kg']['upper']}]"
        )
        for name, body in prediction["prediction"]["bodies"].items()
    }
)


## 3. 视频观测检查

同步查看 RGB、深度、物体 Mask 与部件 Mask。保留模块一的视觉网格，视频不触发额外补面或视觉表面改写。


In [ ]:
video_preview = px.assets.preview_physics_video(video_dir)
show_animation(px.artifacts.get_bytes(video_preview))
geometry_refs = real_geometry["geometries"]
show_metrics({"视频观测": "RGB / Depth / Mask", "视觉几何": "模块一原始重建"})


## 4. 碰撞体与装配

In [ ]:
collision_precision = {
    "body": {"threshold_m": 0.002, "max_parts": 256},
    "cover": {"threshold_m": 0.0025, "max_parts": 256},
}
collision_reports = []
for link in reference["links"]:
    operation = px.assets.build_collision(
        ArtifactRef.model_validate(geometry_refs[link["name"]]),
        method="coacd",
        config={
            "source_unit": "m",
            "link": link["name"],
            **collision_precision[link["name"]],
            "max_volume_error": 0.15,
            "preprocessing": "coacd",
            "weld_tolerance_m": 0.000001,
        },
    )
    collision_reports.append(operation.output("collision_build_report"))
assembly_input = px.artifacts.put_json(
    {
        "schema": "physrealx.reference-assembly-input/1",
        "resource_id": f"laptop1.physical.{run_key}",
        "version": "1.0.0",
        "display_name": "Laptop reconstructed physical asset",
        "reference_model": reference_model.model_dump(mode="json"),
        "reconstructed_geometry": reconstructed_geometry.model_dump(mode="json"),
        "registration": registration.model_dump(mode="json"),
        "predicted_physics": prior.model_dump(mode="json"),
        "collision_reports": [ref.model_dump(mode="json") for ref in collision_reports],
    }
)
assembly = px.assets.assemble(assembly_input, method="reference_asset_assembly").output(
    "asset_assembly"
)
asset = AssetSpec.model_validate(json.loads(px.artifacts.get_bytes(assembly))["asset_spec"])
assert [
    asset.joints[0].lower_limit, asset.joints[0].upper_limit
] == expected_reconstructed_motion["range"]
assert asset.metadata["reconstructed_motion"] == expected_reconstructed_motion


## 5. 铰接碰撞代理修正与仿真检查

保持重建视觉网格和观测关节范围不变，依据原始表面自动细分、重新拟合碰撞代理；同时检查局部误差、接触面覆盖和完整关节范围内的碰撞。


In [ ]:
initial_assembly = assembly
initial_feasibility = px.assets.check_physics_feasibility(
    initial_assembly, config={"sweep_samples": 61}
)
initial_check = json.loads(px.artifacts.get_bytes(initial_feasibility))
articulated_collision_refinement = px.assets.refine_articulated_collision(
    initial_assembly, initial_feasibility,
    config={
        "max_parts_per_link": 768,
        "max_rounds": 3,
        "penetration_safety_margin_m": 5e-5,
    },
)
refinement = json.loads(px.artifacts.get_bytes(articulated_collision_refinement))
if refinement["status"] == "PASSED" and refinement["collision_geometry_refined"]:
    candidate_input = ArtifactRef.model_validate(refinement["candidate_assembly_input"])
    assembly = px.assets.assemble(
        candidate_input, method="reference_asset_assembly"
    ).output("asset_assembly")
    asset = AssetSpec.model_validate(json.loads(px.artifacts.get_bytes(assembly))["asset_spec"])
feasibility = px.assets.check_physics_feasibility(
    assembly, config={"sweep_samples": 361}
)
check = json.loads(px.artifacts.get_bytes(feasibility))
metrics = {
    "初始穿透 (mm)": 1000 * initial_check["max_penetration_m"],
    "仿真可行性": check["status"],
    "候选穿透 (mm)": 1000 * refinement["after"]["max_penetration_m"],
}
workflow_status = (
    "PASSED" if check["status"] == refinement["status"] == "PASSED" else "FAILED"
)
metrics["联合验收"] = workflow_status
metrics["局部表面/接触覆盖"] = refinement["status"]
metrics["视觉网格与位姿"] = "保持不变"
metrics["细分轮次"] = len(refinement["iterations"]) - 1
for link, quality in refinement["fidelity"].items():
    metrics[f"{link} 最大采样偏差 (mm)"] = 1000 * quality["bidirectional_sample_max_m"]
    metrics[f"{link} 最差局部覆盖"] = f"{quality['worst_cell_coverage']:.1%}"
    metrics[f"{link} 接触区域覆盖"] = f"{quality['contact_coverage']:.1%}"
if refinement["status"] != "PASSED":
    metrics["未通过原因"] = refinement["reason"]
show_metrics(metrics)
physical_preview = px.assets.preview_physical_asset(
    assembly, rejected_refinement=articulated_collision_refinement
    if refinement["status"] != "PASSED" else None,
)
report_preview = px.assets.preview_physics_report(
    prior, feasibility, readiness,
    initial_feasibility if refinement["status"] == "PASSED" else None
)
show_animation(px.artifacts.get_bytes(physical_preview))
show_image(px.artifacts.get_bytes(report_preview))

## 6. 导出

In [ ]:
package = px.assets.package(assembly, method="openusd")
usd = package.output("usd_asset_bundle")
package_manifest = package.output("asset_package_manifest")
asset_ref = None
if workflow_status == "PASSED":
    quality = px.assets.validate(asset, profile="simulation")
    assert quality.passed, quality.model_dump(mode="json")
    asset_ref = px.assets.register(asset)
results = {
    "asset": asset_ref.model_dump(mode="json") if asset_ref else None,
    "source_reference": reference_model.model_dump(mode="json"),
    "source_reconstruction": reconstruction_report.model_dump(mode="json"),
    "reconstructed_geometry": reconstructed_geometry.model_dump(mode="json"),
    "prior": prior.model_dump(mode="json"),
    "readiness": readiness.model_dump(mode="json"),
    "initial_assembly": initial_assembly.model_dump(mode="json"),
    "initial_feasibility": initial_feasibility.model_dump(mode="json"),
    "articulated_collision_refinement": articulated_collision_refinement.model_dump(mode="json")
    if articulated_collision_refinement
    else None,
    "assembly": assembly.model_dump(mode="json"),
    "package": package_manifest.model_dump(mode="json"),
    "usd": usd.model_dump(mode="json"),
    "video_preview": video_preview.model_dump(mode="json"),
    "preview": physical_preview.model_dump(mode="json"),
    "feasibility": feasibility.model_dump(mode="json"),
    "workflow_status": workflow_status,
    "geometry_policy": "coordinate_transform_only",
    "independent_physical_validation": False,
}
exports = {
    "registered-reconstruction-geometry.json": reconstructed_geometry,
    "vlm-physics-prior.json": prior,
    "video-readiness.json": readiness,
    "video-mask-manifest.json": ArtifactRef.model_validate(video_status["mask_manifest"]),
    "video-observations.gif": video_preview,
    "initial-assembly.json": initial_assembly,
    "initial-feasibility.json": initial_feasibility,
    "assembly.json": assembly,
    "package.json": package_manifest,
    "physical-asset.usdz": usd,
    "feasibility.json": feasibility,
    "physical-asset-preview.gif": physical_preview,
    "physics-report.png": report_preview,
}
if articulated_collision_refinement is not None:
    exports["articulated-collision-refinement.json"] = articulated_collision_refinement
px.artifacts.export(OUTPUT_DIR, results, files=exports)
print(OUTPUT_DIR / "results.json")